# Carga de dados

In [1]:
import pandas as pd

leitos = pd.read_csv(
    '../data/raw/Leitos_2026.csv',
    sep=';',
    encoding='latin-1'
)
leitos.shape

(43180, 35)

In [2]:
leitos['COMP'].value_counts().sort_index()

COMP
202601    7212
202602    7202
202603    7188
202604    7181
202605    7183
202606    7214
Name: count, dtype: int64

In [3]:
leitos.dtypes

COMP                        int64
REGIAO                        str
UF                            str
CO_IBGE                     int64
MUNICIPIO                     str
MOTIVO_DESABILITACAO      float64
CNES                        int64
NOME_ESTABELECIMENTO          str
RAZAO_SOCIAL                  str
TP_GESTAO                     str
CO_TIPO_UNIDADE             int64
DS_TIPO_UNIDADE               str
NATUREZA_JURIDICA           int64
DESC_NATUREZA_JURIDICA        str
NO_LOGRADOURO                 str
NU_ENDERECO                   str
NO_COMPLEMENTO                str
NO_BAIRRO                     str
CO_CEP                      int64
NU_TELEFONE                   str
NO_EMAIL                      str
LEITOS_EXISTENTES           int64
LEITOS_SUS                  int64
UTI_TOTAL_EXIST             int64
UTI_TOTAL_SUS               int64
UTI_ADULTO_EXIST            int64
UTI_ADULTO_SUS              int64
UTI_PEDIATRICO_EXIST        int64
UTI_PEDIATRICO_SUS          int64
UTI_NEONATAL_E

# Recorte: Minas Gerais

In [4]:
leitos['UF'].value_counts()

UF
SP    6161
MG    3981
BA    3742
RJ    2964
GO    2688
PR    2680
PE    2371
RS    1980
MA    1933
CE    1887
PA    1701
SC    1469
RN    1171
PB    1163
PI    1160
MT    1095
MS     736
ES     681
AM     678
RO     676
AL     588
TO     481
DF     399
SE     333
AC     204
AP     138
RR     120
Name: count, dtype: int64

In [5]:
leitos['MOTIVO_DESABILITACAO'].notna().sum()

np.int64(0)

In [6]:
mg = leitos[leitos['UF'] == 'MG']
mg.shape

(3981, 35)

In [7]:
mg['MUNICIPIO'].nunique()

409

# Concentração por município

In [8]:
jun = mg[mg['COMP'] == 202606]
por_municipio = jun.groupby('MUNICIPIO')['LEITOS_SUS'].sum().sort_values(ascending=False)
por_municipio.head(15)

MUNICIPIO
BELO HORIZONTE          6403
JUIZ DE FORA            1679
UBERLANDIA              1205
MONTES CLAROS            904
TEOFILO OTONI            738
UBERABA                  716
CONTAGEM                 496
IPATINGA                 477
BETIM                    475
GOVERNADOR VALADARES     459
ARAGUARI                 418
BARBACENA                401
MURIAE                   374
PATOS DE MINAS           332
POUSO ALEGRE             310
Name: LEITOS_SUS, dtype: int64

In [9]:
print(por_municipio.head(10).sum() / por_municipio.sum())

0.40324932305769634


# Estrutura dos campos de UTI

In [10]:

jun = mg[mg['COMP'] == 202606]

print((jun['UTI_TOTAL_SUS'] > jun['LEITOS_SUS']).sum())

print(((jun['LEITOS_SUS'] == 0) & (jun['UTI_TOTAL_SUS'] > 0)).sum())

subtipos = (jun['UTI_ADULTO_SUS'] + jun['UTI_PEDIATRICO_SUS'] +
            jun['UTI_NEONATAL_SUS'] + jun['UTI_QUEIMADO_SUS'] +
            jun['UTI_CORONARIANA_SUS'])
print((subtipos != jun['UTI_TOTAL_SUS']).sum())

0
0
0


# Junção com a regionalização de saúde 

In [11]:
regiao = pd.read_excel('../data/raw/Planilha-de-Regionalizacao_Versao-2026.xlsx', skiprows=2, nrows=853)
regiao.columns = ['cod_ibge','cod_datasus','municipio','populacao',
                  'unidade_regional','cod_micro','microrregiao','cod_macro','macrorregiao']
regiao['macrorregiao'] = regiao['macrorregiao'].str.strip()
regiao['microrregiao'] = regiao['microrregiao'].str.strip()
regiao['municipio'] = regiao['municipio'].str.strip()

mg_regiao = mg.merge(regiao, left_on='CO_IBGE', right_on='cod_datasus', how='left')

In [12]:
print(mg_regiao['microrregiao'].isna().sum())

0


# Taxa por microrregião

In [13]:
jun_reg = mg_regiao[mg_regiao['COMP'] == 202606]
pop_macro = regiao.groupby('macrorregiao')['populacao'].sum()
pop_micro = regiao.groupby('microrregiao')['populacao'].sum()

In [14]:
jun_reg = mg_regiao[mg_regiao['COMP'] == 202606]
leitos_micro = jun_reg.groupby('microrregiao')['LEITOS_SUS'].sum()
print(leitos_micro.shape)
print(regiao['populacao'].sum())
print(jun_reg['populacao'].sum())

(89,)
21393441
220675772


In [15]:
leitos_micro = jun_reg.groupby('microrregiao')['LEITOS_SUS'].sum()
taxa = (leitos_micro / pop_micro * 1000).sort_values()
print(taxa.head(10))
print(taxa.tail(10))

microrregiao
VESPASIANO/LAGOA SANTA        0.603687
CORONEL FABRICIANO/TIMÓTEO    0.640897
FRUTAL/ITURAMA                0.669848
PARÁ DE MINAS/NOVA SERRANA    0.691501
CONTAGEM                      0.741443
BETIM                         0.828036
SETE LAGOAS                   0.899825
DIVINÓPOLIS                   0.939608
BOCAIÚVA                      0.942407
JOÃO PINHEIRO                 0.966710
dtype: float64
microrregiao
GUANHÃES                     2.175752
BARBACENA                    2.179374
CARANGOLA                    2.222070
ITAMBACURI                   2.294828
MURIAÉ                       2.425981
SÃO LOURENÇO                 2.453877
JUIZ DE FORA                 2.703460
MANTENA/ITABIRINHA           2.906225
DIAMANTINA/ITAMARANDIBA      3.213275
TEÓFILO OTONI/MALACACHETA    3.335810
dtype: float64


# Taxa por macrorregião

In [16]:
leitos_macro = jun_reg.groupby('macrorregiao')['LEITOS_SUS'].sum()
pop_macro = regiao.groupby('macrorregiao')['populacao'].sum()
taxa_macro = (leitos_macro / pop_macro * 1000).sort_values()
print(taxa_macro)

macrorregiao
VALE DO AÇO           1.151640
EXTREMO SUL           1.159680
OESTE                 1.166321
NOROESTE              1.328730
TRIÂNGULO DO SUL      1.404377
NORTE                 1.468342
CENTRO                1.495403
LESTE DO SUL          1.568500
LESTE                 1.573488
CENTRO SUL            1.592425
TRIÂNGULO DO NORTE    1.697009
SUL                   1.802287
SUDOESTE              1.855231
SUDESTE               2.013505
JEQUITINHONHA         2.216888
NORDESTE              2.409318
dtype: float64


# UTI por macrorregião

In [17]:
uti_macro = jun_reg.groupby('macrorregiao')['UTI_TOTAL_SUS'].sum()
taxa_uti = (uti_macro / pop_macro * 1000).sort_values()
print(taxa_uti)

macrorregiao
NORDESTE              0.078537
NOROESTE              0.124694
LESTE                 0.135830
OESTE                 0.137301
TRIÂNGULO DO SUL      0.138987
NORTE                 0.149414
CENTRO SUL            0.151600
LESTE DO SUL          0.152922
EXTREMO SUL           0.163375
JEQUITINHONHA         0.168403
CENTRO                0.185278
VALE DO AÇO           0.185827
TRIÂNGULO DO NORTE    0.186161
SUL                   0.188121
SUDOESTE              0.201915
SUDESTE               0.239904
dtype: float64


In [18]:
proporcao_uti = (uti_macro / leitos_macro * 100).sort_values()
print(proporcao_uti)

macrorregiao
NORDESTE               3.259727
JEQUITINHONHA          7.596372
LESTE                  8.632396
NOROESTE               9.384460
CENTRO SUL             9.520063
LESTE DO SUL           9.749553
TRIÂNGULO DO SUL       9.896730
NORTE                 10.175725
SUL                   10.437912
SUDOESTE              10.883558
TRIÂNGULO DO NORTE    10.969928
OESTE                 11.772152
SUDESTE               11.914766
CENTRO                12.389824
EXTREMO SUL           14.087948
VALE DO AÇO           16.135881
dtype: float64


# UTI total (rede pública + privada) por macrorregião

In [19]:
uti_total_macro = jun_reg.groupby('macrorregiao')['UTI_TOTAL_EXIST'].sum()

comparacao = pd.DataFrame({
    'uti_sus_10mil': uti_macro / pop_macro * 10000,
    'uti_total_10mil': uti_total_macro / pop_macro * 10000
})
comparacao['percentual_sus'] = comparacao['uti_sus_10mil'] / comparacao['uti_total_10mil'] * 100
print(comparacao.sort_values('uti_total_10mil'))

                    uti_sus_10mil  uti_total_10mil  percentual_sus
macrorregiao                                                      
NORDESTE                 0.785372         0.975381       80.519481
LESTE DO SUL             1.529217         1.936073       78.985507
NORTE                    1.494144         2.046197       73.020528
CENTRO SUL               1.515998         2.067270       73.333333
LESTE                    1.358298         2.090862       64.963504
TRIÂNGULO DO SUL         1.389874         2.199627       63.186813
EXTREMO SUL              1.633751         2.360912       69.200000
NOROESTE                 1.246941         2.386619       52.247191
SUDOESTE                 2.019152         2.442522       82.666667
JEQUITINHONHA            1.684031         2.463209       68.367347
VALE DO AÇO              1.858272         2.738507       67.857143
OESTE                    1.373011         2.768167       49.600000
SUL                      1.881211         3.005613       62.58

# Cobertura de planos de saúde (ANS)

In [20]:
cobertura = pd.read_csv(
    '../data/raw/A12332510_22_1_196.csv',
    sep=';',
    encoding='latin-1',
    skiprows=4,
    decimal=',',
    nrows=853
)
cobertura.columns = ['municipio_raw', 'taxa_cobertura']
print(cobertura.shape)
print(cobertura.tail(3))

(853, 2)
                     municipio_raw  taxa_cobertura
850  317200 Visconde do Rio Branco            19.0
851            317210 Volta Grande            12.7
852          317220 Wenceslau Braz             8.4


In [21]:
cobertura['cod_datasus'] = cobertura['municipio_raw'].str[:6].astype(int)
print(cobertura[['municipio_raw', 'cod_datasus', 'taxa_cobertura']].head(3))
print(cobertura['cod_datasus'].duplicated().sum())

                municipio_raw  cod_datasus  taxa_cobertura
0  310010 Abadia dos Dourados       310010             8.3
1               310020 Abaeté       310020            15.1
2           310030 Abre Campo       310030             5.8
0


# Taxa ajustada pela população dependente do SUS 

In [22]:
regiao_cob = regiao.merge(cobertura[['cod_datasus','taxa_cobertura']],
                          on='cod_datasus', how='left')
print(regiao_cob.shape, regiao_cob['taxa_cobertura'].isna().sum())

regiao_cob['pop_sus'] = regiao_cob['populacao'] * (1 - regiao_cob['taxa_cobertura']/100)
pop_sus_macro = regiao_cob.groupby('macrorregiao')['pop_sus'].sum()

print((uti_macro / pop_sus_macro * 10000).sort_values())

(853, 10) 0
macrorregiao
NORDESTE              0.851286
NOROESTE              1.567588
LESTE                 1.618681
NORTE                 1.648425
LESTE DO SUL          1.744248
JEQUITINHONHA         1.843608
OESTE                 2.002394
CENTRO SUL            2.031192
TRIÂNGULO DO SUL      2.082511
EXTREMO SUL           2.248838
SUDOESTE              2.389969
SUL                   2.483363
TRIÂNGULO DO NORTE    2.608499
VALE DO AÇO           2.699821
CENTRO                3.060634
SUDESTE               3.156315
dtype: float64


In [23]:
taxa_leitos_sus_ajustada = (leitos_macro / pop_sus_macro * 1000).sort_values()
print(taxa_leitos_sus_ajustada)

macrorregiao
EXTREMO SUL           1.596285
NORTE                 1.619958
NOROESTE              1.670408
VALE DO AÇO           1.673179
OESTE                 1.700958
LESTE DO SUL          1.789054
LESTE                 1.875124
TRIÂNGULO DO SUL      2.104242
CENTRO SUL            2.133591
SUDOESTE              2.195945
TRIÂNGULO DO NORTE    2.377864
SUL                   2.379176
JEQUITINHONHA         2.426959
CENTRO                2.470280
NORDESTE              2.611527
SUDESTE               2.649079
dtype: float64


In [24]:
cob_macro = (1 - pop_sus_macro / pop_macro) * 100
print(cob_macro.sort_values())

macrorregiao
NORDESTE               7.742939
JEQUITINHONHA          8.655728
NORTE                  9.359272
LESTE DO SUL          12.327979
SUDOESTE              15.515591
LESTE                 16.086161
NOROESTE              20.454756
SUDESTE               23.992265
SUL                   24.247422
CENTRO SUL            25.364123
EXTREMO SUL           27.351347
TRIÂNGULO DO NORTE    28.633021
VALE DO AÇO           31.170530
OESTE                 31.431520
TRIÂNGULO DO SUL      33.259702
CENTRO                39.464250
dtype: float64
